## 003 - AppleCider Metadata
<a id='index'></a> <br>

- [part 1: import stuff, other basic things](#import)
    - if you haven't already processed the dataset into alerts, or created `data_train.csv`, `test_df.csv`, etc, go back to the previous notebook [001: data pre-processing walkthrough](https://github.com/ajunell/AppleCider/blob/main/notebooks/001-data-processing.ipynb). this won't work unless you've done all the preprocessing steps! 
- [part 2: Dataset](#dataset)

In [1]:
import pandas as pd ; import numpy as np ; import os ; from tqdm.auto import tqdm
import matplotlib.pyplot as plt ; from matplotlib.gridspec import GridSpec
import random ; import torch ; import joblib
import sys
sys.path.insert(0, '/Users/junell/Documents/AppleCider')

from AppleCider.preprocess.data_preprocessor import AlertProcessor, PhotometryProcessor, DataPreprocessor, SpectraProcessor
from AppleCider.preprocess.data_preprocessor import DataSorter
from AppleCider.preprocess.transient_dataset import TransientDataset
import AppleCider.preprocess.plot_data as plot_data

from AppleCider.preprocess.transient_dataset import TransientDataset

from AppleCider.core.model import Informer
from AppleCider.core.trainer import Trainer
from AppleCider.core.dataset import DataGenerator

from datetime import datetime
import optuna 
from optuna.exceptions import DuplicatedStudyError
from math import sqrt
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
import wandb
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from scipy.interpolate import interp1d 
from scipy import stats
import pickle 

In [3]:
wandb.login()

wandb: Currently logged in as: ajunell (ajunell-university-of-minnesota) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

<a id='import'></a><br>

In [2]:
data_dir = '/Users/junell/Documents/AppleCider_Data/SEDM_folder'
SEDM_dataset = pd.read_csv('/Users/junell/Documents/AppleCider/SEDM_dataset.csv')
CLASSES = ['SN Ia', 'SN II', 'SN IIP', 'Cataclysmic', 'AGN', 'SN IIn', 'SN Ic', 'SN Ib', 'SN IIb', 'Tidal Disruption Event',
          'Stellar variable']

TEST_DATA_PATH = '/Users/junell/Documents/AppleCider_Data/data_test'
TRAIN_DATA_PATH = '/Users/junell/Documents/AppleCider_Data/data_train'

id2target = {i: CLASSES[i] for i in range(11)}
target2id = {v: k for k, v in id2target.items()}

data_test = pd.read_csv('/Users/junell/Documents/AppleCider/data_test.csv')
data_train = pd.read_csv('/Users/junell/Documents/AppleCider/data_train.csv')

data_test['type_encoded'] = data_test['type'].map(target2id)
data_train['type_encoded'] = data_train['type'].map(target2id)

In [3]:
file_path= '/Users/junell/Documents/AppleCider/'
filename = os.path.join(file_path, 'test_files.pkl')
filename_train = os.path.join(file_path, 'train_files.pkl')
filename_val = os.path.join(file_path, 'val_files.pkl')

## write to file using pickle
#with open(filename, 'wb') as file:
#    pickle.dump(test_files, file)

#with open(filename_train, 'wb') as file:
#    pickle.dump(train_files, file)
#
#with open(filename_val, 'wb') as file:
#    pickle.dump(val_files, file)

## load files
with open(filename, 'rb') as file:
    test_files = pickle.load(file)
with open(filename_train, 'rb') as file:
    train_files = pickle.load(file)
with open(filename_val, 'rb') as file:
    val_files = pickle.load(file)

<big>part 2: Dataset</big><br>


<a id='dataset'></a><br>
<i><small>[back to index](#index)</small></i>

In [4]:
from torch import nn

class DataGenerator_notebook(torch.utils.data.Dataset):

    def __init__(self, preprocessed_path, df, step, file_list=None, **kwargs):
        super().__init__(**kwargs)
        self.preprocessed_path = preprocessed_path
        self.step = step
        self.df = df
        
        
        self.id2target = {i: x for i, x in enumerate(sorted(self.df[self.step].unique()))}
        self.target2id = {'SN Ia': 0 , 'SN Ic': 0,  'SN Ib': 0, 'SN II': 1, 'SN IIP': 1, 'SN IIn': 1, 'SN IIb': 1,
                          'Cataclysmic': 2, 'AGN': 3, 'Tidal Disruption Event': 4}

        if file_list is not None:
            self.data_files = file_list
        else:
            self.data_files = [f for f in os.listdir(preprocessed_path) if f.endswith('.npy')]
        
    def __len__(self): 
        return(len(self.data_files))
    
    def __getitem__(self, index):    
        ''' load processed object alerts to get photometry, metadata, images''' 
        file_path = os.path.join(self.preprocessed_path, str(self.data_files[index]))
        sample = np.load(file_path, allow_pickle=True).item()
        
        obj_id = sample['obj_id']
        photometry = sample['photometry']
        metadata = sample['metadata'].to_numpy()
        images = sample['images']
        spectra = sample['spectra']

        # get spectra csv, save wavelengths fluxes
        obj_id_alert = str(self.data_files[index])
        obj_id = obj_id_alert[:12] # only includes ZTFID from 'ZTFID_alerts.npy' 
        
        # get label
        obj_df = self.df[self.df['name'] == obj_id]
        obj_label = obj_df['type_encoded'].iloc[0]
        
        # convert photometry, metadata, images, spectra to tensors
        photometry_tensor = torch.tensor(photometry, dtype=torch.float32)
        #padd tensors
        photo_len = len(photometry_tensor)
        max_photo = 225  # maximum photometry length from an alert
        add_dim = max_photo - photo_len
        
        # padded photometry so all photometry the same length
        if photo_len <= 225:
            photometry_padded = nn.ConstantPad1d((0, 0, 0, add_dim), 0)(photometry_tensor)
        else:
            # check max photo length from alerts again! 
            print("too much photometry. try again!", photo_len)
            
        photometry_mask = torch.ones((photometry_padded.size(0), photometry_padded.size(1)))    
            
        #metadata_tensor = torch.tensor(metadata)
        images_tensor = torch.tensor(images)
        spectra_tensor = torch.from_numpy(spectra)
        # convert label to tensor
        target = torch.tensor(obj_label).type(torch.LongTensor)  
        
        # functionally these are blanks
        blank_spectra = torch.zeros((225, 4))
        #blank_metadata = torch.zeros((225, 4))
        blank_images = torch.zeros((225, 4))

        return photometry_padded, photometry_mask, metadata, blank_images, blank_spectra, target

In [5]:
def collate_fn_WORKS_PHOTO(data):
    photometry, _, _,_,_,labels = zip(*data)
    
    labels = torch.tensor(labels, dtype=torch.long)
    
    photometry = torch.stack(photometry)
    photometry_mask = torch.ones((photometry.size(0), photometry.size(1)))
    
    spectra = torch.zeros((len(data), 225))
    metadata = torch.zeros((len(data), 225))
    images = torch.zeros((len(data), 225))
    obj_id = torch.zeros((len(data), 225))
    
    # works like this:
    return photometry, photometry_mask, metadata, images, spectra, labels

In [6]:
train_dataset = DataGenerator_notebook(TRAIN_DATA_PATH, data_train, 'type', file_list=train_files)
val_dataset = DataGenerator_notebook(TRAIN_DATA_PATH, data_train, 'type', file_list=val_files)

_,_, metadata, _,_,_ = train_dataset[50]
metadata.shape,\
metadata.dtype

((10,), dtype('float64'))

In [7]:
metadata = [el[2] for el in train_dataset]
metadata = np.array(metadata)

In [8]:
scaler = StandardScaler()
scaler.fit(metadata)

StandardScaler()

In [9]:
print("Column means:", scaler.mean_)
print("Column standard deviations:", np.sqrt(scaler.var_))

Column means: [-1.64727546e+00 -1.19284498e+01  3.68007088e+00 -3.98625834e+00
  1.86019591e+02  2.66189962e+01  8.78550277e+00  1.43418695e-01
  2.69004963e+01  1.07869205e-01]
Column standard deviations: [ 43.87288939 110.42313242   3.61572184 111.45310653 103.6402131
  25.16502848  45.29058809  11.3412928   22.42123886   1.37878498]


In [37]:
metadata[0]

array([-9.99000000e+02, -9.99000000e+02,  3.29015404e-01, -9.99000000e+02,
        1.86453587e+02,  3.35467715e+01, -9.99000000e+02,  8.29999968e-02,
       -9.99000000e+02,  1.83817698e-03])

In [40]:
scaler.transform(metadata[0].reshape(1,-1))[0]

array([-2.28996822e+01, -8.65813355e+00, -9.20474390e-01, -8.64776871e+00,
        2.36640749e-02,  3.20416796e-01, -2.23775167e+01, -2.89164391e-03,
       -3.84019299e+01, -6.19482092e-02])

In [10]:
def get_metadata_scaler(train_dataset, metadata_scaler_path):
    
    metadata = [el[2] for el in train_dataset]
    metadata = np.array(metadata)
    
    scaler = StandardScaler()
    scaler.fit(metadata)
    
    print("Column means:", scaler.mean_)
    print("Column standard deviations:", np.sqrt(scaler.var_),"\n")
    
    print("save to:", metadata_scaler_path)
    joblib.dump(scaler, os.path.join(metadata_scaler_path,'scaler.pkl'))


In [11]:
get_metadata_scaler(train_dataset, '/Users/junell/Documents/AppleCider/AppleCider/core/')

Column means: [-1.64727546e+00 -1.19284498e+01  3.68007088e+00 -3.98625834e+00
  1.86019591e+02  2.66189962e+01  8.78550277e+00  1.43418695e-01
  2.69004963e+01  1.07869205e-01]
Column standard deviations: [ 43.87288939 110.42313242   3.61572184 111.45310653 103.6402131
  25.16502848  45.29058809  11.3412928   22.42123886   1.37878498] 

save to: /Users/junell/Documents/AppleCider/AppleCider/core/
